# Lab 5: Point Estimation, Normal Conjugacy, and Credible Intervals

**Phân tích dữ liệu Bayesian - IUH**

Sau khi đã có posterior, một câu hỏi mới xuất hiện: ta nên tóm tắt nó như thế nào? Có lúc ta cần một con số duy nhất để ra quyết định. Có lúc ta cần một khoảng để nói cho người khác biết mức độ bất định còn lại lớn đến đâu. Và có lúc chính sự khác nhau giữa mean, median và MAP mới là điều đáng học nhất.

Lab 5 đi thẳng vào những câu hỏi ấy. Xen giữa các bài là những chỗ đề phụ thuộc MSSV, nên ở đây ta khóa một bộ giá trị minh họa để giữ trọng tâm ở lập luận Bayesian: posterior được tạo ra ra sao, rồi được chuyển thành quyết định hay thành khoảng bất định như thế nào.


In [ ]:
# Lab 5 gom nhiều kiểu bài nên cell import này chuẩn bị đủ tích phân, phân phối chuẩn và trực quan hóa.
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.integrate import quad

plt.rcParams["figure.figsize"] = (10, 5)


## 1. Khi posterior phải biến thành quyết định

Nửa đầu của lab xoay quanh các bài toán ước lượng điểm. Cùng một posterior, nhưng nếu tiêu chuẩn mất mát thay đổi thì điểm tóm tắt tối ưu cũng thay đổi theo. Điều đó buộc ta thôi nhìn posterior như một đích đến cuối cùng, và bắt đầu nhìn nó như nguyên liệu cho một quyết định cụ thể.

Nửa sau chuyển sang Normal conjugacy và credible intervals. Ở đây, trọng tâm không còn là chọn một con số duy nhất, mà là diễn tả được mức độ chắc chắn hay bất định còn lại sau dữ liệu. Chính nhịp chuyển từ “một điểm” sang “một khoảng” làm lab này trở thành cầu nối rất tốt giữa suy luận Bayes và giao tiếp kết quả thống kê.


## 2. Bài 1 - Point estimates dưới các hàm tổn thất khác nhau

Với prior $f_X(x)=3x^2$ trên $[0,1]$, ta có Beta$(3,1)$. Mỗi phần của bài hỏi một “điểm tốt nhất” khác nhau, tức là một quyết định tối ưu dưới một hàm tổn thất khác nhau.

### Ý nghĩa của bài tập

Bài này làm rất rõ một thông điệp then chốt của Bayesian decision theory: không có một “ước lượng tốt nhất” tuyệt đối nếu ta chưa nói rõ tiêu chuẩn mất mát. Cùng một posterior, nhưng khi mục tiêu ra quyết định thay đổi thì điểm tóm tắt tối ưu cũng đổi theo.

Điều này rất quan trọng về mặt tư duy. Nó giúp sinh viên thôi tìm một đáp án điểm mang tính thần thánh, và bắt đầu hỏi đúng câu hơn: tôi muốn tối ưu điều gì khi chọn một con số đại diện cho posterior này?


### Đề bài nhắc lại

**Bài 1.** Cho prior $X$ có mật độ $f_X(x)=3x^2$ trên $[0,1]$. Với từng likelihood và quan sát tương ứng sau đây, hãy tìm ước lượng điểm thích hợp:

- a) $Y\mid X=x \sim \text{Geometric}(x)$, tìm MAP khi $y=3$.
- b) $Y\mid X=x \sim \text{Binomial}(5,x)$, tìm ML khi $y=2$.
- c) $Y\mid X=x \sim \text{Exponential}(x)$, tìm MMSE khi $y=1/3$.
- d) $Y\mid X=x$ có mật độ $4xy$ trên $0\le y\le 1$, tìm khoảng tin cậy $70\%$ cho hậu nghiệm khi $y=1/4$.


In [ ]:
# Bài 1 đặt cạnh MAP, ML, MMSE và credible interval để thấy “điểm tốt nhất” phụ thuộc vào mục tiêu ra quyết định.
# (a) MAP with Geometric(y=3): posterior proportional to x(1-x)^2 * 3x^2 = 3x^3(1-x)^2 -> Beta(4,3)?
# Wait: geometric on {1,2,...}: x(1-x)^(y-1), with y=3 gives x(1-x)^2; multiply by x^2 => x^3(1-x)^2 -> Beta(4,3)
map_estimate = (4 - 1) / (4 + 3 - 2)
ml_estimate = 2 / 5

unnorm_mmse = lambda x: 3 * x**3 * np.exp(-x / 3)
den_mmse, _ = quad(unnorm_mmse, 0, 1)
num_mmse, _ = quad(lambda x: x * unnorm_mmse(x), 0, 1)
mmse_estimate = num_mmse / den_mmse

# (d) posterior for f(y|x)=4xy with y=1/4: proportional to x * 3x^2 = 3x^3 -> Beta(4,1)
credible_interval = stats.beta.ppf([0.15, 0.85], 4, 1)

print(f"MAP (part a) = {map_estimate:.4f}")
print(f"ML (part b) = {ml_estimate:.4f}")
print(f"MMSE (part c) = {mmse_estimate:.4f}")
print(f"70% credible interval (part d) = [{credible_interval[0]:.4f}, {credible_interval[1]:.4f}]")


### Interpretation

Bài 1 cho thấy “ước lượng tốt nhất” không có một đáp án duy nhất nếu ta không nói rõ tiêu chuẩn ra quyết định. Dưới mất mát bình phương ta dùng posterior mean, dưới mất mát trị tuyệt đối ta dùng median, còn dưới mất mát 0-1 ta dùng MAP.


## 3. Bài 2 - Hai bài cá nhân hóa, giải bằng một ví dụ minh họa

Bài 2 được cá nhân hóa theo MSSV, nên notebook này cố ý giải bằng một ví dụ minh họa cụ thể nhưng vẫn giữ cấu trúc tổng quát của lời giải. Điều quan trọng không phải là bốn chữ số cụ thể, mà là cách ta biến dữ liệu cá nhân hóa thành các tham số cập nhật trong mô hình liên hợp.

### Ý nghĩa của bài tập

Ý nghĩa của bài này là giúp sinh viên tách biệt giữa con số đầu vào và logic Bayesian đứng sau. Khi đề thay MSSV, lời giải số sẽ thay đổi, nhưng cách xây posterior, cách diễn giải mean và variance hậu nghiệm, và cách cập nhật tuần tự thì không đổi.

Đây cũng là một bài học tốt về cách viết solution notebook cho đề cá nhân hóa: phải nói rõ giả định đang dùng, đồng thời giữ công thức ở dạng đủ tổng quát để người học thay dữ liệu của chính mình vào.


### Đề bài nhắc lại

**Bài 2.** Đây là hai bài cá nhân hóa theo MSSV trong lab gốc:

- 1) Số lỗi trung bình mỗi tuần của một thiết bị mới có prior Gamma với mean $4$ và độ lệch chuẩn $2$. Trong 4 tuần, số lỗi quan sát là bốn số cuối MSSV. Hãy xây dựng posterior qua từng tuần.
- 2) Tỷ lệ tuyển dụng có prior Beta với mean $32\%$ và độ lệch chuẩn $16\%$. Trong 60 sinh viên, tất cả $b$ sinh viên có điểm Bayesian khá/giỏi đều trúng tuyển, với $b$ là tổng chữ số trong MSSV. Hãy tìm posterior và rủi ro hậu nghiệm.

Trong solution, hai bài này được giải bằng một ví dụ minh họa chung để chỉ ra quy trình.


In [ ]:
# Vì đề gốc phụ thuộc MSSV, cell này chốt một ví dụ minh họa cụ thể nhưng vẫn giữ công thức Bayesian tổng quát.
# Example assumption from MSSV-dependent prompt
week_counts = np.array([1, 2, 3, 4])
b = 10

# 2.1 Gamma-Poisson
prior_shape, prior_rate = 4, 1
shapes = [prior_shape]
rates = [prior_rate]
for y in week_counts:
    shapes.append(shapes[-1] + y)
    rates.append(rates[-1] + 1)

print('Gamma-Poisson sequential posteriors:')
for step, (a, r) in enumerate(zip(shapes, rates)):
    label = 'prior' if step == 0 else f'week {step}'
    print(f"  {label}: Gamma({a:.0f}, {r:.0f}), mean={a/r:.4f}")

# 2.2 Beta-Binomial
mean_prior = 0.32
sd_prior = 0.16
var_prior = sd_prior**2
kappa = mean_prior * (1 - mean_prior) / var_prior - 1
alpha_prior = mean_prior * kappa
beta_prior = (1 - mean_prior) * kappa
alpha_post = alpha_prior + b
beta_post = beta_prior + (60 - b)
posterior_risk = alpha_post * beta_post / ((alpha_post + beta_post)**2 * (alpha_post + beta_post + 1))
print(f"Recruitment posterior Beta({alpha_post:.1f}, {beta_post:.1f})")
print(f"Posterior risk (variance) = {posterior_risk:.6f}")


## 4. Bài 3 - Xét nghiệm dương tính rồi nhiều lần âm tính

Bài xét nghiệm này tiếp tục đào sâu một trực giác Bayesian quan trọng: một kết quả dương tính ban đầu chưa kết thúc câu chuyện. Khi thêm các kết quả âm tính tiếp theo, posterior có thể đảo chiều đáng kể vì mỗi phép đo mới lại là một mẩu bằng chứng mới.

### Ý nghĩa của bài tập

Ý nghĩa lớn nhất của bài là chuyển suy nghĩ từ “đếm xem bao nhiêu test dương hay âm” sang “sau mỗi test, niềm tin hợp lý nên là bao nhiêu”. Đây chính là cách Bayes biến chuỗi dữ liệu tuần tự thành chuỗi quyết định tuần tự.

Bài cũng rất gần với thực hành lâm sàng và kiểm tra chất lượng: điều ta cần không chỉ là một kết quả test riêng lẻ, mà là một ngưỡng posterior đủ mạnh để hành động. Vì vậy câu hỏi “cần thêm bao nhiêu lần âm tính?” là một câu hỏi ra quyết định rất tự nhiên.


### Đề bài nhắc lại

**Bài 3.** Một căn bệnh có prior nhiễm là $0.2\%$. Xét nghiệm có độ nhạy $90\%$ và dương tính giả $1\%$. Một người test lần đầu dương tính.

- a) Tính xác suất người này thực sự mắc bệnh.
- b) Nếu sau đó người này tiếp tục test nhiều lần và đều âm tính, cần ít nhất bao nhiêu lần âm tính để xác suất không nhiễm bệnh vượt $99\%$?


In [ ]:
# Bài xét nghiệm: dùng posterior probability thay vì trực giác thô để quyết định cần bao nhiêu lần âm tính tiếp theo.
prevalence = 0.002
sensitivity = 0.9
false_positive = 0.01
false_negative = 0.1
specificity = 1 - false_positive

posterior = sensitivity * prevalence / (sensitivity * prevalence + false_positive * (1 - prevalence))
print(f"P(D | +) = {posterior:.6f}")

neg_tests_needed = 0
while 1 - posterior <= 0.99:
    posterior = false_negative * posterior / (false_negative * posterior + specificity * (1 - posterior))
    neg_tests_needed += 1
print(f"Minimum additional negative tests to get P(no disease) > 0.99: {neg_tests_needed}")


## 5. Bài 4 và Bài 5 - Normal-Normal conjugacy

Bài 4 và Bài 5 được đặt cạnh nhau để làm nổi bật sức mạnh của liên hợp chuẩn. Khi prior và likelihood đều Gaussian, posterior vẫn giữ dạng Gaussian và các công thức cập nhật có thể đọc như sự cân bằng giữa độ tin của prior và lượng thông tin trong dữ liệu.

### Ý nghĩa của bài tập

Ý nghĩa của nhóm bài này là giúp người học thấy Bayesian updating không nhất thiết phải phụ thuộc vào tính toán số nặng. Trong nhiều mô hình cổ điển, ta có thể nhìn rất rõ dữ liệu đã kéo posterior mean đi bao xa và làm posterior variance co lại như thế nào.

Đây cũng là một khuôn mẫu cực kỳ quan trọng cho các mô hình hồi quy Bayesian ở các lab sau, nơi công thức cập nhật ma trận chỉ là phiên bản nhiều chiều của cùng ý tưởng này.


### Đề bài nhắc lại

Phần này gộp hai bài chuẩn liên hợp Normal-Normal:

- **Bài 4.** Prior $X\sim N(16,0.25)$, dữ liệu gồm 10 quan sát chuẩn đã cho sẵn; hãy xác định posterior.
- **Bài 5.** Với prior $X\sim N(0,1)$ và $Y\mid X=x\sim N(x,1)$, hãy chứng minh hậu nghiệm $f_{X\mid Y}(x\mid y)$ là $N(y/2,1/2)$.


In [ ]:
# Hai bài normal conjugacy được gom lại để sinh viên thấy công thức cập nhật làm việc trực tiếp trên mean và variance.
obs = np.array([16.11, 17.37, 16.35, 15.16, 18.82, 18.12, 15.82, 16.34, 16.64, 15.0])
prior_mean = 16
prior_var = 0.25
lik_var = obs.var(ddof=0)
post_var = 1 / (1 / prior_var + len(obs) / lik_var)
post_mean = post_var * (prior_mean / prior_var + obs.sum() / lik_var)
print(f"Posterior for Bài 4: N({post_mean:.4f}, {post_var:.4f})")

# Bài 5 proof check by formula
print("For prior N(0,1) and likelihood N(x,1), posterior is N(y/2, 1/2).")


## 6. Bài 6, 7, 8 - Credible intervals and posterior summaries

Ba bài cuối cùng gom nhiều họ phân phối khác nhau nhưng cùng chung một câu hỏi: posterior nên được báo cáo thế nào để diễn đạt mức độ bất định một cách có ý nghĩa? Khi chuyển từ Normal sang Beta rồi Gamma, ta thấy credible interval là một ý tưởng thống nhất, không bị trói vào một mô hình riêng lẻ.

### Ý nghĩa của bài tập

Ý nghĩa của nhóm bài này là giúp sinh viên vượt qua thói quen chỉ báo cáo posterior mean. Trong thực hành Bayesian, một khoảng đáng tin cậy thường truyền tải nhiều thông tin hơn vì nó cho biết cả vị trí trung tâm lẫn độ rộng của bất định còn lại.

Bài tập cũng tạo nền tốt cho kỹ năng giao tiếp kết quả thống kê: cùng một posterior, ta có thể cần mean, MAP, interval hay xác suất vượt ngưỡng tùy theo câu hỏi của người ra quyết định.


### Đề bài nhắc lại

Phần này gộp ba bài cuối của lab:

- **Bài 6.** Prior $X\sim N(0,4)$, $Y_i\mid X=x\sim N(x,1)$ với $25$ quan sát và $\sum Y_i=14$. Tìm MAP và khoảng tin cậy $99\%$ cho $X\mid \overline{Y}$.
- **Bài 7.** Với prior Beta$(3.3,7.2)$ và dữ liệu $11/27$ sinh viên ngủ ít nhất 6 giờ, hãy xây dựng một tập tin cậy $90\%$ cho tham số tổng thể.
- **Bài 8.** Một công ty điện thoại quan sát $7265$ cuộc gọi trong $10$ giờ, giả sử số cuộc gọi mỗi giờ là Poisson. Hãy ước lượng tần suất ban đầu và xây dựng tập tin cậy $95\%$.


In [ ]:
# Cell cuối cùng tóm tắt ba bài về credible interval và posterior summary dưới các mô hình khác nhau.
# Bài 6
prior_mean_6 = 0
prior_var_6 = 4
n = 25
sum_y = 14
post_var_6 = 1 / (1 / prior_var_6 + n / 1)
post_mean_6 = post_var_6 * sum_y
ci99 = stats.norm.ppf([0.005, 0.995], loc=post_mean_6, scale=math.sqrt(post_var_6))
print(f"Bài 6 posterior mean (also MAP) = {post_mean_6:.4f}")
print(f"Bài 6 99% credible interval = [{ci99[0]:.4f}, {ci99[1]:.4f}]")

# Bài 7
alpha_post_7 = 3.3 + 11
beta_post_7 = 7.2 + 16
ci90 = stats.beta.ppf([0.05, 0.95], alpha_post_7, beta_post_7)
print(f"Bài 7 posterior Beta({alpha_post_7:.1f}, {beta_post_7:.1f})")
print(f"Bài 7 90% credible interval = [{ci90[0]:.4f}, {ci90[1]:.4f}]")

# Bài 8
prior_rate = 1 / 1000
posterior_shape_8 = 1 + 7265
posterior_rate_8 = prior_rate + 10
mean_prior_rate = 1 / prior_rate
ci95_8 = stats.gamma.ppf([0.025, 0.975], a=posterior_shape_8, scale=1/posterior_rate_8)
print(f"Bài 8 prior mean call rate = {mean_prior_rate:.1f} calls/hour")
print(f"Bài 8 posterior mean call rate = {posterior_shape_8/posterior_rate_8:.2f} calls/hour")
print(f"Bài 8 95% credible interval = [{ci95_8[0]:.2f}, {ci95_8[1]:.2f}]")


### Reflection on inconsistencies

Bài 6 của đề gốc ghi gợi ý khoảng tin cậy không khớp với mức 99%. Với posterior variance tính đúng từ mô hình, khoảng 99% rộng hơn đáng kể so với gợi ý đó. Trong notebook này tôi ưu tiên tính nhất quán với công thức Bayes hơn là giữ nguyên đáp án gợi ý có vẻ sai.


## 7. Model criticism and reflection

Lab 5 nhấn mạnh rằng nhiều quyết định trong Bayesian analysis là quyết định về mục tiêu: ta muốn một điểm đại diện, một posterior đầy đủ, hay một khoảng đáng tin cậy? Ngoài ra, các bài phụ thuộc MSSV cho thấy một solution notebook tốt phải nói rõ giả định đã chọn, nếu không người học dễ lẫn lộn giữa công thức tổng quát và con số ví dụ.


## 8. Conceptual interpretation questions

1. Vì sao MAP, ML và MMSE có thể cho ba đáp án khác nhau trên cùng một bài?  
2. Posterior variance nên được hiểu như thế nào khi đề bài hỏi “rủi ro hậu nghiệm”?  
3. Ở Bài 6, vì sao credible interval phụ thuộc trực tiếp vào posterior variance?  
4. Khi một đề cá nhân hóa theo MSSV, solution notebook nên xử lý thế nào để vẫn dạy được lập luận Bayesian?


## 9. Final takeaway

Lab 5 gom nhiều kỹ thuật khác nhau, nhưng tất cả đều quay về một điểm: posterior không chỉ cho ta một con số, mà cho ta một phân phối để từ đó chọn ra quyết định, khoảng tin cậy và diễn giải mức bất định phù hợp với câu hỏi đang đặt ra.


## 10. Lecture references

Nếu muốn tự học mượt hơn theo từng cụm bài của Lab 5, nên quay lại các lecture sau:

- [Bài 2.4: Posterior - Cập nhật Niềm tin với Dữ liệu](../../contents/vi/chapter02/_posts/2025-01-02-02_04_posterior.md): **Bài 1**, **Bài 3** và **Bài 6-8**: đọc lại để nhớ rằng mọi point estimate, posterior probability hay credible interval đều xuất phát từ cùng một posterior.
- [Bài 2.5: Prior liên hợp và đại số của cập nhật Bayes](../../contents/vi/chapter02/_posts/2025-01-02-02_05_conjugate_priors.md): **Bài 2**, **Bài 4** và **Bài 5**: đọc lại phần Normal conjugacy và các cập nhật đóng dạng để theo kịp các bài có prior-likelihood liên hợp hoặc dữ liệu phụ thuộc MSSV.
- [Bài 8.4: Bayesian Decision Analysis - From Inference to Action](../../contents/vi/chapter08/_posts/2025-01-05-08_04_decision_analysis.md): **Bài 1** và **Bài 3**: đặc biệt cần khi muốn hiểu vì sao posterior mean, median, MAP hay xác suất vượt ngưỡng là các cách ra quyết định khác nhau.
- Gợi ý ôn tập thêm: với **Bài 6**, **Bài 7** và **Bài 8**, hãy đối chiếu lại cách diễn giải credible interval trong [Bài 2.4: Posterior - Cập nhật Niềm tin với Dữ liệu](../../contents/vi/chapter02/_posts/2025-01-02-02_04_posterior.md) để giữ đúng tinh thần bất định hậu nghiệm.
